In [1]:
# !pip install pandas --upgrade  --quiet
# !pip install numpy --upgrade  --quiet
# !pip install scipy --upgrade  --quiet
# !pip install statsmodels --upgrade  --quiet
# !pip install scikit-learn --upgrade  --quiet
# !pip install missingno --upgrade  --quiet
!pip install apafib --upgrade --quiet

Objetivos de aprendizaje:
1. Hacer un mínimo análisis exploratorio de un conjunto de datos
2. Hacer el preproceso de un conjunto de datos para usar regresión
3. Saber plantear problemas de regresión sencillos y resolverlos usando diferentes
métodos
4. Interpretar los resultados de un problema de regresión

# Problema 1
El coste de los seguros médicos varía bastante según las circunstancias de cada persona, pero a veces averiguar como se calcula realmente no es tan sencillo. El conjunto de datos Medical Cost Personal Dataset tiene la descripción de las características de un grupo de personas y los cargos de su seguro médico. Nos interesa predecir esta última variable ($charges$).
Trabajaremos con una versión de este conjunto que podéis obtener mediante la función `load_medical_costs` de la librería `apafib`.

In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import normaltest
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.impute import KNNImputer
from sklearn.preprocessing import Binarizer, MinMaxScaler, StandardScaler, PowerTransformer

import missingno as msno

from apafib import load_medical_costs

medical = load_medical_costs()
medical.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## Apartat a)
 - Dividir el conjunt en (70%/30%) i fer una exploració mínima del conjunt de dades.
 - Aplicar PCA a les dades i representar la variable resposta en els dos primers components.
 - Transformar les variables per poder ajustar un model de regressió.

 ---

### Dividir el conjunt i exploració inicial


In [3]:
medical_train, medical_test = train_test_split(medical, test_size=0.3, random_state=92)

print("Diviri el conjunt de dades en entrenament i test en una proporció 70/30:")
print("-"*70)
print(f"Mida conjunt entrenament: {medical_train.shape}")
print(f"Mida conjunt test: {medical_test.shape}")
print()

Diviri el conjunt de dades en entrenament i test en una proporció 70/30:
----------------------------------------------------------------------
Mida conjunt entrenament: (936, 7)
Mida conjunt test: (402, 7)



### Comprovar valors perduts
Per poder decidir que fer amb el processament de dades, hem de mirar si existeix algun valor perdut, ja que en el cas que hi hagin hauriem de decidir com tractasr-los, si eliminar-los o substituir el seu valor.

In [4]:
medical_train.isna().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Veiem que no existeix cap valor perdut, així que no ens hem de preocupar per tractar-los.
Examinem les característiques del dataset, i així podrem identificar les variables numériques i les catagòriques

In [5]:
medical.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,1338.0,NaN,NaN,NaN,39.207025,14.04996,18.0,27.0,39.0,51.0,64.0
sex,1338,2,male,676,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bmi,1338.0,NaN,NaN,NaN,30.663397,6.098187,15.96,26.29625,30.4,34.69375,53.13
children,1338.0,NaN,NaN,NaN,1.094918,1.205493,0.0,0.0,1.0,2.0,5.0
smoker,1338,2,no,1064,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,1338,4,southeast,364,NaN,NaN,NaN,NaN,NaN,NaN,NaN
charges,1338.0,NaN,NaN,NaN,13270.422265,12110.011237,1121.8739,4740.28715,9382.033,16639.912515,63770.42801


Podem veure que les variables categòriques son : {$sex$, $smoker$, $region$}
I per tant les númeriques les restants : {$age$, $bmi$, $children$} i evidentment la nostre variable objectiu $charges$.

Ara observarem la distribució de les variables categòriques: 

In [6]:
print(medical_train['sex'].value_counts())
print('-' * 90)
print(medical_train['smoker'].value_counts())
print('-' * 90)
print(medical_train['region'].value_counts())

sex
male      477
female    459
Name: count, dtype: int64
------------------------------------------------------------------------------------------
smoker
no     747
yes    189
Name: count, dtype: int64
------------------------------------------------------------------------------------------
region
southeast    252
northwest    239
northeast    223
southwest    222
Name: count, dtype: int64
